In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [2]:
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.7
)

In [4]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [5]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [6]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [7]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer=checkpointer)

In [8]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'Football'}, config=config1)

{'topic': 'Football',
 'joke': "Why did the football coach go to the bank?\nBecause he wanted his *quarterback*!\n\n(Okay, okay, that's American Football. Here's a proper one for *soccer* football!)\n\nMy friend still doesn't understand the offside rule.\nHe thinks it's when the players just get *too far ahead of themselves*!",
 'explanation': 'Okay, let\'s break down these two football (American and Soccer) jokes!\n\n---\n\n### Joke 1: Why did the football coach go to the bank?\n**Because he wanted his *quarterback*!**\n\n**Explanation:**\n\n1.  **"Quarterback" (American Football):** In American football, the "quarterback" is a specific, very important player position. They are usually the leader of the offense, responsible for calling plays and throwing the ball.\n2.  **"Quarter Back" (Bank Context):** At a bank (or any store), if you\'re getting change, you might ask for a "quarter back" – meaning you want a 25-cent coin returned to you.\n3.  **The Pun:** The humor comes from the pl

In [9]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'Football', 'joke': "Why did the football coach go to the bank?\nBecause he wanted his *quarterback*!\n\n(Okay, okay, that's American Football. Here's a proper one for *soccer* football!)\n\nMy friend still doesn't understand the offside rule.\nHe thinks it's when the players just get *too far ahead of themselves*!", 'explanation': 'Okay, let\'s break down these two football (American and Soccer) jokes!\n\n---\n\n### Joke 1: Why did the football coach go to the bank?\n**Because he wanted his *quarterback*!**\n\n**Explanation:**\n\n1.  **"Quarterback" (American Football):** In American football, the "quarterback" is a specific, very important player position. They are usually the leader of the offense, responsible for calling plays and throwing the ball.\n2.  **"Quarter Back" (Bank Context):** At a bank (or any store), if you\'re getting change, you might ask for a "quarter back" – meaning you want a 25-cent coin returned to you.\n3.  **The Pun:** The humo

In [10]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'Football', 'joke': "Why did the football coach go to the bank?\nBecause he wanted his *quarterback*!\n\n(Okay, okay, that's American Football. Here's a proper one for *soccer* football!)\n\nMy friend still doesn't understand the offside rule.\nHe thinks it's when the players just get *too far ahead of themselves*!", 'explanation': 'Okay, let\'s break down these two football (American and Soccer) jokes!\n\n---\n\n### Joke 1: Why did the football coach go to the bank?\n**Because he wanted his *quarterback*!**\n\n**Explanation:**\n\n1.  **"Quarterback" (American Football):** In American football, the "quarterback" is a specific, very important player position. They are usually the leader of the offense, responsible for calling plays and throwing the ball.\n2.  **"Quarter Back" (Bank Context):** At a bank (or any store), if you\'re getting change, you might ask for a "quarter back" – meaning you want a 25-cent coin returned to you.\n3.  **The Pun:** The hum

In [11]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'Cricket'}, config=config2)

{'topic': 'Cricket',
 'joke': 'Why did the cricket ball break up with the bat?\n\nIt felt like it was always getting *hit on*!',
 'explanation': 'This is a classic pun that plays on the double meaning of the phrase "getting hit on"!\n\nHere\'s the breakdown:\n\n1.  **Literal Meaning (Cricket Context):** In the game of cricket, the entire point of the bat is to *hit* the ball. So, literally, the cricket ball is constantly "getting hit" by the bat.\n\n2.  **Idiomatic Meaning (Social Context):** The phrase "to hit on someone" is an idiom that means to flirt with them, make romantic advances, or try to pick them up. If someone feels like they are "always getting hit on," it means they are constantly receiving unwanted or excessive flirtatious attention.\n\n**The Joke:**\n\nThe humor comes from anthropomorphizing the cricket ball (giving it human feelings and motivations). The joke pretends the ball broke up with the bat for a human relationship reason – because it felt like it was always r

In [12]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket ball break up with the bat?\n\nIt felt like it was always getting *hit on*!', 'explanation': 'This is a classic pun that plays on the double meaning of the phrase "getting hit on"!\n\nHere\'s the breakdown:\n\n1.  **Literal Meaning (Cricket Context):** In the game of cricket, the entire point of the bat is to *hit* the ball. So, literally, the cricket ball is constantly "getting hit" by the bat.\n\n2.  **Idiomatic Meaning (Social Context):** The phrase "to hit on someone" is an idiom that means to flirt with them, make romantic advances, or try to pick them up. If someone feels like they are "always getting hit on," it means they are constantly receiving unwanted or excessive flirtatious attention.\n\n**The Joke:**\n\nThe humor comes from anthropomorphizing the cricket ball (giving it human feelings and motivations). The joke pretends the ball broke up with the bat for a human relationship reason – because it felt l

In [13]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket ball break up with the bat?\n\nIt felt like it was always getting *hit on*!', 'explanation': 'This is a classic pun that plays on the double meaning of the phrase "getting hit on"!\n\nHere\'s the breakdown:\n\n1.  **Literal Meaning (Cricket Context):** In the game of cricket, the entire point of the bat is to *hit* the ball. So, literally, the cricket ball is constantly "getting hit" by the bat.\n\n2.  **Idiomatic Meaning (Social Context):** The phrase "to hit on someone" is an idiom that means to flirt with them, make romantic advances, or try to pick them up. If someone feels like they are "always getting hit on," it means they are constantly receiving unwanted or excessive flirtatious attention.\n\n**The Joke:**\n\nThe humor comes from anthropomorphizing the cricket ball (giving it human feelings and motivations). The joke pretends the ball broke up with the bat for a human relationship reason – because it felt 

### Time Travel

In [14]:
workflow.get_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f15d969-8ed6-683c-8000-0c7a2f030ccd"}})

StateSnapshot(values={}, next=(), config={'configurable': {'thread_id': '2', 'checkpoint_id': '1f15d969-8ed6-683c-8000-0c7a2f030ccd'}}, metadata=None, created_at=None, parent_config=None, tasks=(), interrupts=())

In [16]:
workflow.invoke(None, {"configurable": {"thread_id": "2", "checkpoint_id": "1f15d969-8ed6-683c-8000-0c7a2f030ccd"}})

EmptyInputError: Received no input for __start__

In [17]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket ball break up with the bat?\n\nIt felt like it was always getting *hit on*!', 'explanation': 'This is a classic pun that plays on the double meaning of the phrase "getting hit on"!\n\nHere\'s the breakdown:\n\n1.  **Literal Meaning (Cricket Context):** In the game of cricket, the entire point of the bat is to *hit* the ball. So, literally, the cricket ball is constantly "getting hit" by the bat.\n\n2.  **Idiomatic Meaning (Social Context):** The phrase "to hit on someone" is an idiom that means to flirt with them, make romantic advances, or try to pick them up. If someone feels like they are "always getting hit on," it means they are constantly receiving unwanted or excessive flirtatious attention.\n\n**The Joke:**\n\nThe humor comes from anthropomorphizing the cricket ball (giving it human feelings and motivations). The joke pretends the ball broke up with the bat for a human relationship reason – because it felt 

#### Updating State

In [18]:
workflow.update_state({"configurable": {"thread_id": "2", "checkpoint_id": "1f15d969-8ed6-683c-8000-0c7a2f030ccd", "checkpoint_ns": ""}}, {'topic':'Tenis'})

{'configurable': {'thread_id': '2',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1d95dd-f3e5-6362-8000-8991506740b1'}}

In [19]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'Tenis'}, next=('generate_joke',), config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f1d95dd-f3e5-6362-8000-8991506740b1'}}, metadata={'source': 'update', 'step': 0, 'parents': {}}, created_at='2026-11-05T21:14:22.578569+00:00', parent_config={'configurable': {'thread_id': '2', 'checkpoint_ns': '', 'checkpoint_id': '1f15d969-8ed6-683c-8000-0c7a2f030ccd'}}, tasks=(PregelTask(id='a1ca6613-f482-bb70-9117-2c0b84188c5f', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'Cricket', 'joke': 'Why did the cricket ball break up with the bat?\n\nIt felt like it was always getting *hit on*!', 'explanation': 'This is a classic pun that plays on the double meaning of the phrase "getting hit on"!\n\nHere\'s the breakdown:\n\n1.  **Literal Meaning (Cricket Context):** In the game of cricket, the entire point of the bat is t

### Fault Tolerance

In [20]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [21]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [22]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(30)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [23]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [24]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)
✅ Step 3 executed


In [25]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)


🔁 Re-running the graph to demonstrate fault tolerance...

✅ Final State: {'input': 'start', 'step1': 'done', 'step2': 'done'}


In [26]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))

[StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=(), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d95df-aaa0-64c7-8003-e7ff2e2767b1'}}, metadata={'source': 'loop', 'step': 3, 'parents': {}}, created_at='2026-11-05T21:15:08.582829+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d95df-aaa0-64c6-8002-0c84651f8b20'}}, tasks=(), interrupts=()),
 StateSnapshot(values={'input': 'start', 'step1': 'done', 'step2': 'done'}, next=('step_3',), config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d95df-aaa0-64c6-8002-0c84651f8b20'}}, metadata={'source': 'loop', 'step': 2, 'parents': {}}, created_at='2026-11-05T21:15:08.582829+00:00', parent_config={'configurable': {'thread_id': 'thread-1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d95de-8c83-6844-8001-bb4aa0002b8c'}}, tasks=(PregelTask(id='be49372c-6050-02af-ee15-20078135ac9